# 09 · Inspect the project database

Run the first Airflow DAG before querying the tables.

## Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.settings import engine

db = engine()


## Batches

In [ ]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM raw.batches ORDER BY group_number, batch_number'), conn))


## Raw and clean counts

In [ ]:
with db.connect() as conn:
    for table in ['raw.properties', 'clean.properties', 'audit.inference_events']:
        print(table, conn.execute(text(f'SELECT count(*) FROM {table}')).scalar())


## Decisions and model history

In [ ]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM audit.batch_decisions ORDER BY batch_number'), conn))
    display(pd.read_sql_query(text('SELECT * FROM audit.training_events ORDER BY id'), conn))


## Sample predictions

In [ ]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT created_at, prediction, model_version, pod, status, error FROM audit.inference_events ORDER BY id DESC LIMIT 10'), conn))


## Training data coverage

In [ ]:
with db.connect() as conn:
    display(pd.read_sql_query(text('''
        WITH available AS (
            SELECT group_number, count(*) AS clean_rows
            FROM clean.properties GROUP BY group_number
        ), champions AS (
            SELECT DISTINCT ON (group_number) group_number, batch_number, model_version, (to_jsonb(t)->>'train_rows')::int AS train_rows
            FROM audit.training_events t WHERE promoted
            ORDER BY group_number, batch_number DESC
        )
        SELECT a.group_number, a.clean_rows, c.batch_number, c.model_version, c.train_rows,
               round(100.0 * c.train_rows / nullif(a.clean_rows, 0), 1) AS train_pct_of_current_clean
        FROM available a LEFT JOIN champions c USING (group_number)
    '''), conn))
db.dispose()


## Notes

Run these read-only queries after processing a batch and again after serving predictions. Capture batch lineage, the decision and model comparison, and the inference audit.
